### Validation of Metadata via SHACL Shapes


Ziel ist es die gefetchten Metadaten durch pyshacl zu validieren durch bereit gestellte SHACL-Shapes

In [ ]:
from pathlib import Path
from rdflib import Graph, OWL
from pyshacl import validate

SHAPES_DIR = Path("~/masterthesis/master_thesis_prototype/shacl_shapes").expanduser().resolve()
ONT_CACHE = SHAPES_DIR / "_ont_cache.ttl"

# Profil: DCAT-AP.de 2.0 - Spezifikation
SHAPES_FILES = [
    SHAPES_DIR / "dcat-ap-de-imports.ttl",
    SHAPES_DIR / "dcat-ap-spec-german-messages.ttl",
    SHAPES_DIR / "dcat-ap-spec-german-additions.ttl",
    SHAPES_DIR / "dcat-ap-de-deprecated.ttl",
    SHAPES_DIR / "dcat-ap_2.1.1_shacl_shapes.ttl",
]

shapes_graph = Graph()
for f in SHAPES_FILES:
    shapes_graph.parse(f)
print(f"Shapes: {len(shapes_graph)} Triples")

# Ontologie-Cache: einmal laden, dann lokal gecacht
ont_graph = Graph()
if ONT_CACHE.exists():
    ont_graph.parse(ONT_CACHE, format="ttl")
    print(f"Ontologie aus Cache geladen: {len(ont_graph)} Triples")
else:
    import_urls = [str(o) for _, _, o in shapes_graph.triples((None, OWL.imports, None))]
    print(f"Lade {len(import_urls)} externe Vokabularien (einmalig)...")
    for url in import_urls:
        try:
            ont_graph.parse(url)
            print(f"  OK: {url}")
        except Exception as e:
            print(f"  WARN: {url} -> {e}")
    ont_graph.serialize(ONT_CACHE, format="ttl")
    print(f"Cache gespeichert: {ONT_CACHE} ({len(ont_graph)} Triples)")


In [ ]:
DATA_DIR = Path("~/masterthesis/master_thesis_prototype/data/sample_2026-06-18_13-45").expanduser().resolve()

data_files = sorted(DATA_DIR.glob("*.rdf"))
if not data_files:
    raise FileNotFoundError(f"Keine .rdf Dateien in {DATA_DIR}")

for data_file in data_files:
    data_graph = Graph()
    data_graph.parse(data_file, format="xml")

    conforms, results_graph, results_text = validate(
        data_graph=data_graph,
        shacl_graph=shapes_graph,
        ont_graph=ont_graph,
        inference=None,
        abort_on_first=False,
        allow_infos=False,
        allow_warnings=False,
        meta_shacl=False,
        advanced=False,
        js=False,
        debug=False,
    )

    print("=" * 80)
    print(f"File: {data_file.name}  |  Conforms: {conforms}")
    print("=" * 80)
    if not conforms:
        print(results_text)
    print()
